# 03 - Measurement-Count Distribution Analysis

This notebook documents how many measurement files are available per fabric sample in the existing NIR-SORT split. It separates all microscopy files, all spectroscopy files, and the selected spectroscopy representation used later in spectral preprocessing. It is descriptive dataset documentation only: it does not change the split, preprocessing, model training, or evaluation logic.

## Setup

Only metadata CSV files are read. The notebook does not open microscopy image files or spectra CSV contents.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

SPLITS = ["train", "val", "test"]
SELECTED_SPECTRA = {
    "modality_group": "spectra",
    "modality": "Benchtop_N",
    "instrument": "N",
    "data_format": "A",
}
SELECTED_SPECTRA_LABEL = "Benchtop_N / N/A"
PLOT_COLOURS = {
    "Microscopy images": "#2563eb",
    "All spectroscopy": "#7c3aed",
    "Selected spectra": "#059669",
}

pd.set_option("display.max_columns", 80)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Split Metadata

The split CSVs already contain the fabric-level split assignment and the measurement manifest rows. `Sample ID` is normalised to `sample_id` so the sample table and manifest can be joined consistently.

In [ ]:
def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "mdpp_v2" / "data_splits").exists():
            return candidate / "mdpp_v2"
        if (candidate / "data_splits").exists() and candidate.name == "mdpp_v2":
            return candidate
    raise FileNotFoundError("Could not locate mdpp_v2/data_splits from the current working directory.")


MDPP_ROOT = resolve_mdpp_root()
DATA_SPLITS_DIR = MDPP_ROOT / "data_splits"
ARTIFACTS_DIR = MDPP_ROOT / "artifacts"
MEASUREMENT_DISTRIBUTION_PLOT = ARTIFACTS_DIR / "measurement_count_distribution.png"

sample_tables = []
manifest_tables = []
for split in SPLITS:
    sample_path = DATA_SPLITS_DIR / split / "samples.csv"
    manifest_path = DATA_SPLITS_DIR / split / "measurements_manifest.csv"
    samples_split = pd.read_csv(sample_path).rename(columns={"Sample ID": "sample_id"})
    manifest_split = pd.read_csv(manifest_path)
    samples_split["split"] = split
    manifest_split["split"] = split
    sample_tables.append(samples_split)
    manifest_tables.append(manifest_split)

samples = pd.concat(sample_tables, ignore_index=True)
manifest = pd.concat(manifest_tables, ignore_index=True)

input_summary = pd.DataFrame([
    {
        "source": "fabric samples",
        "rows": len(samples),
        "unique_fabrics": samples["sample_id"].nunique(),
        "splits": ", ".join(SPLITS),
    },
    {
        "source": "measurement manifest",
        "rows": len(manifest),
        "unique_fabrics": manifest["sample_id"].nunique(),
        "splits": ", ".join(SPLITS),
    },
])
input_summary

## 2. Count Files Per Fabric

Microscopy availability is counted across all image records in the current split manifests. Spectroscopy availability is counted in two ways: all spectroscopy records for the Data Splitting section, and the selected `Benchtop_N / N/A` absorbance subset for the later Spectral Preprocessing section.

In [ ]:
image_records = manifest[manifest["modality_group"].eq("image")].copy()
all_spectra_records = manifest[manifest["modality_group"].eq("spectra")].copy()
selected_spectra_records = manifest[
    manifest["modality_group"].eq(SELECTED_SPECTRA["modality_group"])
    & manifest["modality"].eq(SELECTED_SPECTRA["modality"])
    & manifest["instrument"].eq(SELECTED_SPECTRA["instrument"])
    & manifest["data_format"].eq(SELECTED_SPECTRA["data_format"])
].copy()

fabric_counts = samples[[
    "sample_id",
    "split",
    "split_stratum",
    "cnn_label",
    "cnn_eligible",
    "classification_label",
]].copy()
fabric_counts["image_files"] = (
    fabric_counts["sample_id"].map(image_records.groupby("sample_id").size()).fillna(0).astype(int)
)
fabric_counts["all_spectra_files"] = (
    fabric_counts["sample_id"].map(all_spectra_records.groupby("sample_id").size()).fillna(0).astype(int)
)
fabric_counts["selected_spectra_files"] = (
    fabric_counts["sample_id"].map(selected_spectra_records.groupby("sample_id").size()).fillna(0).astype(int)
)

measurement_record_summary = pd.DataFrame([
    {
        "measurement_type": "Microscopy images",
        "section_use": "Data Splitting",
        "filter": "modality_group == image",
        "files": len(image_records),
        "fabrics": image_records["sample_id"].nunique(),
    },
    {
        "measurement_type": "All spectroscopy",
        "section_use": "Data Splitting",
        "filter": "modality_group == spectra",
        "files": len(all_spectra_records),
        "fabrics": all_spectra_records["sample_id"].nunique(),
    },
    {
        "measurement_type": "Selected spectra",
        "section_use": "Spectral Preprocessing",
        "filter": SELECTED_SPECTRA_LABEL,
        "files": len(selected_spectra_records),
        "fabrics": selected_spectra_records["sample_id"].nunique(),
    },
])
measurement_record_summary

## 3. Overall Measurement Availability

The first table gives descriptive statistics per fabric. The second table gives the number and percentage of fabric samples for each observed file count. The selected-spectra row is kept separate from the all-spectroscopy row because it belongs to the later spectral-preprocessing choice.

In [ ]:
def count_summary(count_column: str, label: str, section_use: str) -> dict[str, object]:
    values = fabric_counts[count_column]
    return {
        "measurement_type": label,
        "section_use": section_use,
        "fabric_samples": int(values.size),
        "total_files": int(values.sum()),
        "min_files_per_fabric": int(values.min()),
        "median_files_per_fabric": float(values.median()),
        "mean_files_per_fabric": float(values.mean()),
        "max_files_per_fabric": int(values.max()),
        "fabrics_with_0_files": int(values.eq(0).sum()),
        "fabrics_with_1_file": int(values.eq(1).sum()),
        "fabrics_with_2_files": int(values.eq(2).sum()),
        "fabrics_with_1_or_2_files": int(values.isin([1, 2]).sum()),
    }


def distribution_table(count_column: str, label: str, section_use: str) -> pd.DataFrame:
    counts = fabric_counts[count_column].value_counts().sort_index()
    table = counts.rename_axis("files_per_fabric").reset_index(name="fabric_samples")
    table.insert(0, "measurement_type", label)
    table.insert(1, "section_use", section_use)
    table["percentage"] = (table["fabric_samples"] / len(fabric_counts) * 100).round(1)
    return table


overall_summary = pd.DataFrame([
    count_summary("image_files", "Microscopy images", "Data Splitting"),
    count_summary("all_spectra_files", "All spectroscopy", "Data Splitting"),
    count_summary("selected_spectra_files", "Selected spectra", "Spectral Preprocessing"),
])
distribution_summary = pd.concat([
    distribution_table("image_files", "Microscopy images", "Data Splitting"),
    distribution_table("all_spectra_files", "All spectroscopy", "Data Splitting"),
    distribution_table("selected_spectra_files", "Selected spectra", "Spectral Preprocessing"),
], ignore_index=True)

display(overall_summary)
display(distribution_summary)

## 4. Split-Level Availability

This table checks whether the train, validation, and test splits have comparable measurement availability per fabric. The microscopy and all-spectroscopy columns support the Data Splitting section, while the selected-spectra columns support the later spectral-preprocessing documentation.

In [ ]:
split_availability = (
    fabric_counts.groupby("split", sort=False)
    .agg(
        fabric_samples=("sample_id", "nunique"),
        image_files=("image_files", "sum"),
        image_min=("image_files", "min"),
        image_median=("image_files", "median"),
        image_mean=("image_files", "mean"),
        image_max=("image_files", "max"),
        all_spectra_files=("all_spectra_files", "sum"),
        all_spectra_min=("all_spectra_files", "min"),
        all_spectra_median=("all_spectra_files", "median"),
        all_spectra_mean=("all_spectra_files", "mean"),
        all_spectra_max=("all_spectra_files", "max"),
        selected_spectra_files=("selected_spectra_files", "sum"),
        selected_spectra_min=("selected_spectra_files", "min"),
        selected_spectra_median=("selected_spectra_files", "median"),
        selected_spectra_mean=("selected_spectra_files", "mean"),
        selected_spectra_max=("selected_spectra_files", "max"),
    )
    .reset_index()
)
for column in ["image_mean", "all_spectra_mean", "selected_spectra_mean"]:
    split_availability[column] = split_availability[column].round(2)
split_availability

## 5. Class / Stratum Availability

The supervisor question also mentions classes. This table uses `split_stratum` instead of `classification_label` so the `EXCLUDED_CNN` fabrics remain visible and the full set of 155 fabrics is represented.

In [ ]:
stratum_availability = (
    fabric_counts.groupby("split_stratum", sort=False)
    .agg(
        fabric_samples=("sample_id", "nunique"),
        image_min=("image_files", "min"),
        image_median=("image_files", "median"),
        image_mean=("image_files", "mean"),
        image_max=("image_files", "max"),
        all_spectra_min=("all_spectra_files", "min"),
        all_spectra_median=("all_spectra_files", "median"),
        all_spectra_mean=("all_spectra_files", "mean"),
        all_spectra_max=("all_spectra_files", "max"),
        selected_spectra_min=("selected_spectra_files", "min"),
        selected_spectra_median=("selected_spectra_files", "median"),
        selected_spectra_mean=("selected_spectra_files", "mean"),
        selected_spectra_max=("selected_spectra_files", "max"),
    )
    .sort_values("fabric_samples", ascending=False)
    .reset_index()
)
for column in ["image_mean", "all_spectra_mean", "selected_spectra_mean"]:
    stratum_availability[column] = stratum_availability[column].round(2)
stratum_availability

## 6. Low-Measurement Check

This answers the concrete question of whether some fabric samples have only one or two measurement files. The all-spectroscopy check is the relevant spectroscopy check for Data Splitting; the selected-spectra check is only for the later selected representation.

In [ ]:
low_measurement_summary = pd.DataFrame([
    {
        "measurement_type": "Microscopy images",
        "section_use": "Data Splitting",
        "fabrics_with_1_file": int(fabric_counts["image_files"].eq(1).sum()),
        "fabrics_with_2_files": int(fabric_counts["image_files"].eq(2).sum()),
        "fabrics_with_1_or_2_files": int(fabric_counts["image_files"].isin([1, 2]).sum()),
        "minimum_files_per_fabric": int(fabric_counts["image_files"].min()),
    },
    {
        "measurement_type": "All spectroscopy",
        "section_use": "Data Splitting",
        "fabrics_with_1_file": int(fabric_counts["all_spectra_files"].eq(1).sum()),
        "fabrics_with_2_files": int(fabric_counts["all_spectra_files"].eq(2).sum()),
        "fabrics_with_1_or_2_files": int(fabric_counts["all_spectra_files"].isin([1, 2]).sum()),
        "minimum_files_per_fabric": int(fabric_counts["all_spectra_files"].min()),
    },
    {
        "measurement_type": "Selected spectra",
        "section_use": "Spectral Preprocessing",
        "fabrics_with_1_file": int(fabric_counts["selected_spectra_files"].eq(1).sum()),
        "fabrics_with_2_files": int(fabric_counts["selected_spectra_files"].eq(2).sum()),
        "fabrics_with_1_or_2_files": int(fabric_counts["selected_spectra_files"].isin([1, 2]).sum()),
        "minimum_files_per_fabric": int(fabric_counts["selected_spectra_files"].min()),
    },
])
low_measurement_summary

## 7. Measurement-Count Distribution Figure

This figure shows how many measurement files each fabric sample has. The x-axis is the number of files available for one fabric, and the y-axis is the number of fabric samples with that file count.

The three panels answer slightly different questions:

- **Microscopy images**: how many image files each fabric has.
- **All spectroscopy**: how many spectroscopy files each fabric has in the raw split metadata.
- **Selected spectra**: how many `Benchtop_N / N/A` spectra each fabric has after choosing the spectra representation used later in the prototype.

The selected-spectra panel is therefore not a replacement for the all-spectroscopy panel. It describes the subset used for spectral preprocessing and retrieval.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
plot_specs = [
    ("image_files", "Microscopy images", "Microscopy image files per fabric"),
    ("all_spectra_files", "All spectroscopy", "All spectroscopy files per fabric"),
    ("selected_spectra_files", "Selected spectra", f"Selected {SELECTED_SPECTRA_LABEL} files per fabric"),
]

for ax, (count_column, label, title) in zip(axes, plot_specs):
    section = "Spectral Preprocessing" if label == "Selected spectra" else "Data Splitting"
    table = distribution_table(count_column, label, section)
    ax.bar(
        table["files_per_fabric"].astype(str),
        table["fabric_samples"],
        color=PLOT_COLOURS[label],
        alpha=0.86,
        edgecolor="#111827",
        linewidth=0.6,
    )
    for x, y, pct in zip(table["files_per_fabric"].astype(str), table["fabric_samples"], table["percentage"]):
        ax.text(x, y + 1.0, f"{int(y)}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=7)
    ax.set_title(title)
    ax.set_xlabel("Measurement files per fabric sample")
    ax.set_ylabel("Fabric samples")
    ax.grid(axis="y", alpha=0.2)
    ax.set_ylim(0, max(table["fabric_samples"]) * 1.25)
    ax.tick_params(axis="x", labelrotation=35 if len(table) > 6 else 0)

fig.savefig(MEASUREMENT_DISTRIBUTION_PLOT, dpi=200, bbox_inches="tight")
plt.show()

## 8. Result Summary

The measurement counts show that availability differs per fabric sample. Across the full split, the dataset contains **1,040 microscopy image files** and **4,238 spectroscopy files** for **155 fabric samples**.

For microscopy, each fabric has **4 to 12 image files**, with a median of **6**. For all spectroscopy records, each fabric has **2 to 127 spectroscopy files**, with a median of **42**. This supports the fabric-level split: measurements from the same fabric should stay together, instead of splitting individual files independently.

The selected `Benchtop_N / N/A` spectra subset is smaller than the full spectroscopy set. It contains **717 selected spectra files**. In this selected subset, fabrics have **1 to 7 files**, with a median of **7**. This selected-spectra count belongs to the spectral-preprocessing choice, not to the full spectroscopy availability count.
